<a href="https://colab.research.google.com/github/Marco6-3/deeplearning-notes/blob/main/Hongyi_LeeDeepLearning/HW03/Copy_of_HW03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Homework 3 - Convolutional Neural Network**

This is the example code of homework 3 of the machine learning course by Prof. Hung-yi Lee.

In this homework, you are required to build a convolutional neural network for image classification, possibly with some advanced training tips.


There are three levels here:

**Easy**: Build a simple convolutional neural network as the baseline. (2 pts)

**Medium**: Design a better architecture or adopt different data augmentations to improve the performance. (2 pts)

**Hard**: Utilize provided unlabeled data to obtain better results. (2 pts)

## **About the Dataset**

The dataset used here is food-11, a collection of food images in 11 classes.

For the requirement in the homework, TAs slightly modified the data.
Please DO NOT access the original fully-labeled training data or testing labels.

Also, the modified dataset is for this course only, and any further distribution or commercial use is forbidden.

In [ ]:
# Download the dataset
# You may choose where to download the data.

# Google Drive
!gdown --id '1awF7pZ9Dz7X1jn1_QAiKN-_v56veCEKy' --output food-11.zip

# Dropbox
# !wget https://www.dropbox.com/s/m9q6273jl3djall/food-11.zip -O food-11.zip

# MEGA
# !sudo apt install megatools
# !megadl "https://mega.nz/#!zt1TTIhK!ZuMbg5ZjGWzWX1I6nEUbfjMZgCmAgeqJlwDkqdIryfg"

# Unzip the dataset.
# This may take some time.
!unzip -q food-11.zip

## **Import Packages**

First, we need to import packages that will be used later.

In this homework, we highly rely on **torchvision**, a library of PyTorch.

In [ ]:
# Import necessary packages.
import csv
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image, ImageFile
# "ConcatDataset" and "Subset" are possibly useful when doing semi-supervised learning.
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from torchvision.datasets import DatasetFolder

# This is for the progress bar.
from tqdm.auto import tqdm

# Be tolerant to partially-decoded images in dataloader workers.
ImageFile.LOAD_TRUNCATED_IMAGES = True


## **Dataset, Data Loader, and Transforms**

Torchvision provides lots of useful utilities for image preprocessing, data wrapping as well as data augmentation.

Here, since our data are stored in folders by class labels, we can directly apply **torchvision.datasets.DatasetFolder** for wrapping data without much effort.

Please refer to [PyTorch official website](https://pytorch.org/vision/stable/transforms.html) for details about different transforms.

In [ ]:
# It is important to do data augmentation in training.
# However, not every augmentation is useful.
# Please think about what kind of augmentation is helpful for food recognition.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfm = transforms.Compose([
    # Resize the image into a fixed shape (height = width = 128)
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    # ToTensor() should be before normalization.
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# We don't need random augmentations in testing and validation.
# All we need here is to resize the PIL image and transform it into Tensor.
test_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


In [ ]:
# Batch size for training, validation, and testing.
# A greater batch size usually gives a more stable gradient.
# But the GPU memory is limited, so please adjust it carefully.
batch_size = 128

# Colab stability first: too many workers may crash with shared-memory limits.
if torch.cuda.is_available():
    num_workers = 0
    pin_memory = True
else:
    num_workers = 0
    pin_memory = False


def _load_image(path):
    # Close file handle explicitly to avoid worker-side resource issues.
    with Image.open(path) as img:
        return img.convert("RGB")


# Construct datasets.
# The argument "loader" tells how torchvision reads the data.
train_set = DatasetFolder("food-11/training/labeled", loader=_load_image, extensions="jpg", transform=train_tfm)
valid_set = DatasetFolder("food-11/validation", loader=_load_image, extensions="jpg", transform=test_tfm)
unlabeled_set = DatasetFolder("food-11/training/unlabeled", loader=_load_image, extensions="jpg", transform=train_tfm)
test_set = DatasetFolder("food-11/testing", loader=_load_image, extensions="jpg", transform=test_tfm)

# Construct data loaders.
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

print(f"DataLoader config: num_workers={num_workers}, pin_memory={pin_memory}, batch_size={batch_size}")


## **Model**

The basic model here is simply a stack of convolutional layers followed by some fully-connected layers.

Since there are three channels for a color image (RGB), the input channels of the network must be three.
In each convolutional layer, typically the channels of inputs grow, while the height and width shrink (or remain unchanged, according to some hyperparameters like stride and padding).

Before fed into fully-connected layers, the feature map must be flattened into a single one-dimensional vector (for each image).
These features are then transformed by the fully-connected layers, and finally, we obtain the "logits" for each class.

### **WARNING -- You Must Know**
You are free to modify the model architecture here for further improvement.
However, if you want to use some well-known architectures such as ResNet50, please make sure **NOT** to load the pre-trained weights.
Using such pre-trained models is considered cheating and therefore you will be punished.
Similarly, it is your responsibility to make sure no pre-trained weights are used if you use **torch.hub** to load any modules.

For example, if you use ResNet-18 as your model:

model = torchvision.models.resnet18(pretrained=**False**) → This is fine.

model = torchvision.models.resnet18(pretrained=**True**)  → This is **NOT** allowed.

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        # input image size: [3, 128, 128]
        self.cnn_layers = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2, 0),

            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2, 0),

            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(4, 4, 0),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(256 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, 11),
        )

    def forward(self, x):
        # input (x): [batch_size, 3, 128, 128]
        # output: [batch_size, 11]
        x = self.cnn_layers(x)
        x = x.flatten(1)
        x = self.fc_layers(x)
        return x


## **Training**

You can finish supervised learning by simply running the provided code without any modification.

The function "get_pseudo_labels" is used for semi-supervised learning.
It is expected to get better performance if you use unlabeled data for semi-supervised learning.
However, you have to implement the function on your own and need to adjust several hyperparameters manually.

For more details about semi-supervised learning, please refer to [Prof. Lee's slides](https://speech.ee.ntu.edu.tw/~tlkagk/courses/ML_2016/Lecture/semi%20(v3).pdf).

Again, please notice that utilizing external data (or pre-trained model) for training is **prohibited**.

In [ ]:
class PseudoLabelDataset(Dataset):
    def __init__(self, base_dataset, indices, pseudo_labels):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.pseudo_labels = list(pseudo_labels)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image_idx = self.indices[idx]
        img, _ = self.base_dataset[image_idx]
        return img, self.pseudo_labels[idx]


def get_pseudo_labels(dataset, model, threshold=0.90, batch_size=128, device=None):
    # This function generates pseudo-labels for unlabeled data.
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    pseudo_pin_memory = device.startswith("cuda")
    data_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=pseudo_pin_memory,
    )

    was_training = model.training
    model.eval()

    selected_indices = []
    selected_labels = []
    sample_idx = 0

    with torch.no_grad():
        for img, _ in tqdm(data_loader, desc="Pseudo Labeling", leave=False):
            logits = model(img.to(device))
            probs = torch.softmax(logits, dim=-1)
            confidences, preds = probs.max(dim=-1)

            keep_mask = confidences >= threshold
            if keep_mask.any():
                local_indices = torch.arange(img.size(0))[keep_mask.cpu()] + sample_idx
                selected_indices.extend(local_indices.tolist())
                selected_labels.extend(preds[keep_mask].cpu().tolist())

            sample_idx += img.size(0)

    if was_training:
        model.train()

    print(f"Pseudo labels kept: {len(selected_indices)} / {len(dataset)} (threshold={threshold:.2f})")
    return PseudoLabelDataset(dataset, selected_indices, selected_labels)


In [ ]:
def set_seed(seed=3407):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_one_epoch(model, loader, criterion, optimizer, device, max_norm=10):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for imgs, labels in tqdm(loader, desc="Train", leave=False):
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=-1) == labels).sum().item()
        total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


def valid_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Valid", leave=False):
            imgs = imgs.to(device)
            labels = labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=-1) == labels).sum().item()
            total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


def predict(model, loader, device):
    model.eval()
    predictions = []

    with torch.no_grad():
        for imgs, _ in tqdm(loader, desc="Predict", leave=False):
            logits = model(imgs.to(device))
            predictions.extend(logits.argmax(dim=-1).cpu().tolist())

    return predictions


# "cuda" only when GPUs are available.
device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(3407)

# Initialize a model, and put it on the device specified.
model = Classifier().to(device)

# For the classification task, we use cross-entropy as the measurement of performance.
criterion = nn.CrossEntropyLoss()

# Stronger baseline: larger model capacity + longer training.
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)

# Experiment presets:
# Run A: run_name = "runA", n_epochs = 12, do_semi = False
# Run B: run_name = "runB_fix", n_epochs = 60, do_semi = False
# Run C: run_name = "runC_fix", n_epochs = 60, do_semi = True
run_name = "runB_fix"
n_epochs = 60
scheduler = None

# Semi-supervised settings.
do_semi = False
pseudo_threshold = 0.90
semi_start_epoch = 20

best_valid_acc = -1.0
best_epoch = -1
best_ckpt_path = "best.ckpt"

for epoch in range(n_epochs):
    epoch_id = epoch + 1
    epoch_train_loader = train_loader

    if do_semi and epoch_id >= semi_start_epoch:
        pseudo_set = get_pseudo_labels(
            unlabeled_set,
            model,
            threshold=pseudo_threshold,
            batch_size=batch_size,
            device=device,
        )
        if len(pseudo_set) > 0:
            concat_dataset = ConcatDataset([train_set, pseudo_set])
            epoch_train_loader = DataLoader(
                concat_dataset,
                batch_size=batch_size,
                shuffle=True,
                num_workers=num_workers,
                pin_memory=pin_memory,
            )
            print(f"[ Epoch {epoch_id:03d} ] Semi-supervised samples: {len(pseudo_set)}")
        else:
            print(f"[ Epoch {epoch_id:03d} ] No pseudo labels passed threshold.")

    train_loss, train_acc = train_one_epoch(
        model, epoch_train_loader, criterion, optimizer, device, max_norm=10
    )
    valid_loss, valid_acc = valid_one_epoch(model, valid_loader, criterion, device)

    lr = optimizer.param_groups[0]["lr"]
    print(
        f"[ Epoch {epoch_id:03d}/{n_epochs:03d} ] "
        f"lr = {lr:.6e}, train_loss = {train_loss:.5f}, train_acc = {train_acc:.5f}, "
        f"valid_loss = {valid_loss:.5f}, valid_acc = {valid_acc:.5f}"
    )

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_epoch = epoch_id
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch_id,
                "valid_acc": valid_acc,
            },
            best_ckpt_path,
        )
        print(f"[ Epoch {epoch_id:03d} ] Saved new best checkpoint to {best_ckpt_path}")

    if scheduler is not None:
        scheduler.step()

print(f"Training finished. best_valid_acc = {best_valid_acc:.5f} at epoch {best_epoch:03d}")


## **Testing**

For inference, we need to make sure the model is in eval mode, and the order of the dataset should not be shuffled ("shuffle=False" in test_loader).

Last but not least, don't forget to save the predictions into a single CSV file.
The format of CSV file should follow the rules mentioned in the slides.

### **WARNING -- Keep in Mind**

Cheating includes but not limited to:
1.   using testing labels,
2.   submitting results to previous Kaggle competitions,
3.   sharing predictions with others,
4.   copying codes from any creatures on Earth,
5.   asking other people to do it for you.

Any violations bring you punishments from getting a discount on the final grade to failing the course.

It is your responsibility to check whether your code violates the rules.
When citing codes from the Internet, you should know what these codes exactly do.
You will **NOT** be tolerated if you break the rule and claim you don't know what these codes do.


In [ ]:
# Make sure the model is in eval mode.
# Some modules like Dropout or BatchNorm affect if the model is in training mode.
if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=device)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
        print(
            f"Loaded checkpoint from {best_ckpt_path} "
            f"(epoch={ckpt.get('epoch', 'NA')}, valid_acc={ckpt.get('valid_acc', 'NA')})"
        )
    else:
        model.load_state_dict(ckpt)
        print(f"Loaded state dict from {best_ckpt_path}")
else:
    print(f"Checkpoint {best_ckpt_path} not found. Using current model weights.")

# Predict labels for test set.
predictions = predict(model, test_loader, device)
print(f"Total predictions: {len(predictions)}")


In [ ]:
def infer_submission_header(sample_submission_path=None, default_header="Category"):
    candidates = []
    if sample_submission_path is not None:
        candidates.append(sample_submission_path)
    candidates.extend([
        "sample_submission.csv",
        os.path.join("food-11", "sample_submission.csv"),
    ])

    for path in candidates:
        if not os.path.exists(path):
            continue
        with open(path, "r", newline="") as f:
            reader = csv.reader(f)
            header = next(reader, None)

        if header == ["Id", "Category"]:
            print(f"Detected submission header from {path}: Id,Category")
            return "Category"
        if header == ["Id", "Class"]:
            print(f"Detected submission header from {path}: Id,Class")
            return "Class"

    print(f"sample_submission.csv not found or unsupported. Fallback to Id,{default_header}")
    return default_header


def write_submission(preds, out_path, header_name):
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Id", header_name])
        for i, pred in enumerate(preds):
            writer.writerow([i, int(pred)])


def sanity_check_submission(path, num_test_samples, header_name):
    with open(path, "r", newline="") as f:
        rows = list(csv.reader(f))

    assert rows[0] == ["Id", header_name], f"Unexpected header: {rows[0]}"
    assert len(rows) == num_test_samples + 1, (
        f"Unexpected row count: got {len(rows)}, expected {num_test_samples + 1}"
    )

    for expected_id, row in enumerate(rows[1:]):
        assert len(row) == 2, f"Row format invalid at line {expected_id + 2}: {row}"
        assert int(row[0]) == expected_id, (
            f"Id is not continuous at line {expected_id + 2}: got {row[0]}, expected {expected_id}"
        )

    print(
        f"CSV sanity check passed: header=Id,{header_name}, "
        f"rows={len(rows)}, id_range=0..{num_test_samples - 1}"
    )


# Save predictions into the file with format checks.
submission_header = infer_submission_header(default_header="Category")
if best_epoch > 0:
    submission_path = f"hw03_{run_name}_best_ep{best_epoch:03d}.csv"
else:
    submission_path = f"hw03_{run_name}.csv"

write_submission(predictions, submission_path, submission_header)
sanify_rows = len(test_set)
sanity_check_submission(submission_path, sanify_rows, submission_header)

# Keep a stable default output name as well.
write_submission(predictions, "predict.csv", submission_header)

print("Submission preview:")
with open(submission_path, "r", newline="") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.strip())
